<h1 style=\"text-align: center; font-size: 50px;\"> 🤖 MLFlow Registration for Agentic RAG Model</h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Define the Agentic RAG Model
- Register the Model to MLFlow
- Log Results to MLFlow

# Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2025-10-20 15:56:53 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet 

Note: you may need to restart the kernel to use updated packages.
CPU times: user 99.2 ms, sys: 60.4 ms, total: 160 ms
Wall time: 4.8 s


In [4]:
from __future__ import annotations

import json
import os
import sys
import warnings
from collections import namedtuple
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import pandas as pd
import tensorrt_llm

import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.tracking import MlflowClient
from mlflow.types import ColSpec, DataType, Schema

from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, AutoModelForCausalLM,AutoModel

import torch
import numpy as np
sys.path.append("../src")

# ─────── TRT-LLM ───────
import tensorrt_llm
parent_dir = os.path.dirname(os.path.abspath('.'))
sys.path.append(parent_dir)
from src.trt_llm_langchain import TensorRTLangchain

# Import the new MLflow integration layer
from src.mlflow import Logger

# Import the new MLflow integration layer
from src.mlflow import Logger

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[TensorRT-LLM] TensorRT-LLM version: 0.18.0


# Configure Settings

In [5]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [6]:
# ------------------------- MLflow Experiment Configuration -------------------------
MODEL_NAME = "Agentic_RAG_Model"
RUN_NAME = f"Register_{MODEL_NAME}_Run"
EXPERIMENT_NAME = "Agentic_RAG_Experiment"
DEMO_PATH = "../demo"

# Load configuration
import yaml
with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Define the Agentic RAG Model

In [7]:
# The complex RagAgenticModel class has been extracted to src/mlflow/model.py
# Registration is now handled by the generic Logger class

# Get model path from config
model_path = config.get("model_path")
NEMOTRON_DIR = model_path  # Use config-based path

print(f"Using model path from config: {model_path}")
print("RagAgenticModel business logic has been extracted to src/mlflow/model.py")

Using model path from config: /root/.cache/huggingface/hub/models--nvidia--Llama-3.1-Nemotron-Nano-8B-v1/snapshots/a22e1c57330633cd3522903f9bb82480bf3192a6
RagAgenticModel business logic has been extracted to src/mlflow/model.py


# Register the Model to MLFlow

In [8]:
# 1. Set MLflow tracking URI and experiment
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

Using MLflow tracking URI: /phoenix/mlflow
Experiment: Agentic_RAG_Experiment


In [9]:
# Create MLflow signature for the model
input_schema = Schema([
    ColSpec(DataType.string, name="query")
])

output_schema = Schema([
    ColSpec(DataType.string, name="answer"),
    ColSpec(DataType.string, name="retrieved_chunks"),  # Will be serialized as JSON
    ColSpec(DataType.string, name="messages")           # Will be serialized as JSON
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("Created MLflow signature for model")

Created MLflow signature for model


In [10]:
%%time

# 2. Start an MLflow run and log + register the model using new architecture
with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"Started MLflow run: {run.info.run_id}")

    # Use the new Logger class for model registration with signature
    # Explicitly disable ONNX export to avoid parameter conflicts
    Logger.log_model(
        signature=signature, 
        artifact_path=MODEL_NAME, 
        config_path="../configs/config.yaml",
    )

    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

Started MLflow run: 10ae8c85a9e5475982d1fbddfcae60b2


/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025-10-20 15:57:44.735825624 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Loading Model: [1/3]	Downloading HF model
Downloaded model to /root/.cache/huggingface/hub/models--nvidia--Llama-3.1-Nemotron-Nano-8B-v1/snapshots/54641c1611fcff44fa4865626462445e0a153fc7
Time: 0.581s
Lo

[TensorRT-LLM] TensorRT-LLM version: 0.18.0
[TensorRT-LLM][INFO] Engine version 0.18.0 found in the config file, assuming engine(s) built by new builder API.
[TensorRT-LLM][INFO] Refreshed the MPI local session
[TensorRT-LLM][INFO] MPI size: 1, MPI local size: 1, rank: 0
[TensorRT-LLM][INFO] Rank 0 is using GPU 0
[TensorRT-LLM][WARNING] Fix optionalParams : KV cache reuse disabled because model was not built with paged context FMHA support
[TensorRT-LLM][INFO] TRTGptModel maxNumSequences: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBatchSize: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBeamWidth: 1
[TensorRT-LLM][INFO] TRTGptModel maxSequenceLen: 131072
[TensorRT-LLM][INFO] TRTGptModel maxDraftLen: 0
[TensorRT-LLM][INFO] TRTGptModel mMaxAttentionWindowSize: (131072) * 32
[TensorRT-LLM][INFO] TRTGptModel enableTrtOverlap: 0
[TensorRT-LLM][INFO] TRTGptModel normalizeLogProbs: 0
[TensorRT-LLM][INFO] TRTGptModel maxNumTokens: 8192
[TensorRT-LLM][INFO] TRTGptModel maxInputLen: 8192 = min(maxSeque

2025-10-20 15:59:58 - INFO - 🔧 Generating ONNX model(s) for specified models...
2025-10-20 15:59:58 - INFO - 🔄 Converting transformers model: sentence_transformers_model
2025-10-20 15:59:58 - INFO - 📁 Model directory: sentence_transformers_model
2025-10-20 15:59:58 - INFO - 🔍 Model identified as: transformers
2025-10-20 15:59:58 - INFO - 🤗 Converting loaded Transformers model for task: feature-extraction with opset 17
2025-10-20 16:00:04 - INFO - ✅ Transformers model exported to: sentence_transformers_model/model.onnx
2025-10-20 16:00:04 - INFO - ✅ Converted sentence_transformers_model to directory: sentence_transformers_model
2025-10-20 16:00:04 - INFO - 📦 Added model directory artifact: model_directory -> sentence_transformers_model
2025-10-20 16:00:04 - INFO -   No Triton structure requested, using model directories as-is
2025-10-20 16:00:10 - INFO - Model logged with artifacts: ['model_directory']
2025-10-20 16:00:10 - INFO - ✅ Model logged with model directory created!
Registered 

✅ Model 'Agentic_RAG_Model' successfully logged and registered under experiment 'Agentic_RAG_Experiment'.
CPU times: user 1min 28s, sys: 45.9 s, total: 2min 14s
Wall time: 2min 58s


In [11]:
# 3. Retrieve the latest version from the Model Registry
client = MlflowClient()
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
latest_version = versions[0].version

model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")
print(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
print(f"Signature: {model_info.signature}")

Latest registered version of 'Agentic_RAG_Model': 7
Signature: inputs: 
  ['query': string (required)]
outputs: 
  ['answer': string (required), 'retrieved_chunks': string (required), 'messages': string (required)]
params: 
  None



# Log Results to MLFlow

In [12]:
%%time

# 4. Load the model from the Model Registry
loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
print(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Loading Model: [1/3]	Downloading HF model
Downloaded model to /root/.cache/huggingface/hub/models--nvidia--Llama-3.1-Nemotron-Nano-8B-v1/snapshots/54641c1611fcff44fa4865626462445e0a153fc7


[TensorRT-LLM][INFO] Refreshed the MPI local session


Time: 15.401s
Loading Model: [2/3]	Loading HF model to memory
230it [00:00, 355.85it/s]
Time: 1.442s
Loading Model: [3/3]	Building TRT-LLM engine
Time: 106.957s
Loading model done.
Total latency: 123.802s


[TensorRT-LLM] TensorRT-LLM version: 0.18.0
[TensorRT-LLM][INFO] Engine version 0.18.0 found in the config file, assuming engine(s) built by new builder API.
[TensorRT-LLM][INFO] Refreshed the MPI local session
[TensorRT-LLM][INFO] MPI size: 1, MPI local size: 1, rank: 0
[TensorRT-LLM][INFO] Rank 0 is using GPU 0
[TensorRT-LLM][WARNING] Fix optionalParams : KV cache reuse disabled because model was not built with paged context FMHA support
[TensorRT-LLM][INFO] TRTGptModel maxNumSequences: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBatchSize: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBeamWidth: 1
[TensorRT-LLM][INFO] TRTGptModel maxSequenceLen: 131072
[TensorRT-LLM][INFO] TRTGptModel maxDraftLen: 0
[TensorRT-LLM][INFO] TRTGptModel mMaxAttentionWindowSize: (131072) * 32
[TensorRT-LLM][INFO] TRTGptModel enableTrtOverlap: 0
[TensorRT-LLM][INFO] TRTGptModel normalizeLogProbs: 0
[TensorRT-LLM][INFO] TRTGptModel maxNumTokens: 8192
[TensorRT-LLM][INFO] TRTGptModel maxInputLen: 8192 = min(maxSeque

In [13]:
# 5. Run a sample inference using the loaded model
sample_query = "What is the hardware requirement for AI Studio?"
input_payload = {"query": sample_query}

print("\n=== Running Sample Inference ===")
result = loaded_model.predict(input_payload)


=== Running Sample Inference ===


Processed requests: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Processed requests: 100%|██████████| 1/1 [00:00<00:00,  6.10it/s]


In [14]:
# 6. Print results
print(f"Query:")
print("{sample_query}\n")
print("\n==============\n")

print("Answer:")
print(result.get("answer", "<no answer>"), "\n")
print("\n==============\n")

print("Retrieved Chunks:")
for idx, chunk in enumerate(result.get("retrieved_chunks", []), start=1):
    print(f"  {idx}. {chunk[:100]}{'...' if len(chunk)>100 else ''}")

print("\n==============\n")
print("\nMessage History:")
for msg in result.get("messages", []):
    role = msg.get("role", "<unknown>")
    content = msg.get("content", "")
    print(f"  [{role}]: {content}")

Query:
{sample_query}



Answer:
I don't know. 



Retrieved Chunks:



Message History:
  [user]: What is the hardware requirement for AI Studio?
  [developer]: Relevance check result:
  [assistant]: Yes
  [developer]: Rewritten query:
  [assistant]: The required hardware for AI Studio must have at least X GB of RAM and a multi-core processor. (Assuming specific technical details were missing in your note.)
  [developer]: Generated answer:
  [assistant]: I don't know.


In [15]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-10-20 16:03:28 - INFO - ⏱️ Total execution time: 6m 34.90s
2025-10-20 16:03:28 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).